In [8]:
# 1. Mount your Google Drive to the session
from google.colab import drive
drive.mount('/content/drive')

# 2. Copy the binary array from root Drive to local Colab scratch disk
!cp /content/drive/MyDrive/cloud_ready_football.npz /content/

print("🚀 Data successfully staged onto fast local storage!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Data successfully staged onto fast local storage!


In [11]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class ColabFootballDataset(Dataset):
    """
    Memory-efficient PyTorch Dataset reading directly from a local memory-mapped
    NumPy archive file on Colab scratch disk.
    """
    def __init__(self, npz_path='/content/cloud_ready_football.npz', split='train', seq_len=15):
        # Open in memory-mapped mode to prevent loading the 1GB+ archive fully into RAM
        self.archive = np.load(npz_path, mmap_mode='r')
        self.X = self.archive[f'X_{split}']
        self.y = self.archive[f'y_{split}']
        self.seq_len = seq_len

        # Sliding window size calculation
        self.length = len(self.X) - seq_len + 1

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        # Read sliding sequence context directly from fast disk
        seq_x = self.X[idx:idx + self.seq_len]
        # Target corresponds to the outcome of the last action in the sequence
        target_y = self.y[idx + self.seq_len - 1]

        return torch.tensor(seq_x, dtype=torch.float32), torch.tensor(target_y, dtype=torch.float32)

# Create instances using the fast local file path
train_dataset = ColabFootballDataset(npz_path='/content/cloud_ready_football.npz', split='train', seq_len=15)
val_dataset = ColabFootballDataset(npz_path='/content/cloud_ready_football.npz', split='val', seq_len=15)

# High-throughput batching tailored for cloud VRAM
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Data loaders successfully defined!")
print(f"Training batches per epoch: {len(train_loader)}")
print(f"Validation batches per epoch: {len(val_loader)}")

✅ Data loaders successfully defined!
Training batches per epoch: 3587
Validation batches per epoch: 281


In [12]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class FootballTransformer(nn.Module):
    def __init__(self, input_dim=54, d_model=64, nhead=4, num_layers=2, dim_feedforward=128, dropout=0.3):
        super().__init__()
        self.input_projection = nn.Linear(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_layer = nn.Linear(d_model, 2) # Outputs: [Scoring Prob, Conceding Prob]

    def forward(self, x):
        # x shape: [Batch, Seq_Len, Features]
        x = self.input_projection(x)
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        # Pull the representation from the last action in the sequence window
        out = self.output_layer(x[:, -1, :])
        return torch.sigmoid(out)

In [13]:
import torch.optim as optim
import time
import json

# 1. Initialize GPU Execution Hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

# 2. Instantiate Model and Hyperparameters
model = FootballTransformer(input_dim=54, d_model=64, nhead=4, num_layers=2, dropout=0.3).to(device)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)

# FIX: Removed the 'verbose=True' argument to match current PyTorch API specifications
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

NUM_EPOCHS = 15
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')

# 3. Start High-Throughput Training Loop
for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()
    model.train()
    running_train_loss = 0.0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad(set_to_none=True) # Memory footprint protection
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * batch_x.size(0)

    epoch_train_loss = running_train_loss / len(train_dataset)

    # Validation Phase
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            predictions = model(batch_x)
            loss = criterion(predictions, batch_y)
            running_val_loss += loss.item() * batch_x.size(0)

    epoch_val_loss = running_val_loss / len(val_dataset)
    scheduler.step(epoch_val_loss)

    # Save Loss Metrics Trajectory
    history['train_loss'].append(epoch_train_loss)
    history['val_loss'].append(epoch_val_loss)

    elapsed = time.time() - start_time

    # Custom Logging Vector: Grabs current LR directly from optimizer group data
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss: {epoch_train_loss:.5f} | Val Loss: {epoch_val_loss:.5f} | LR: {current_lr:.6f} | Time: {elapsed:.1f}s")

    # Save the Best Model Weights State Checkpoint back to mounted Drive
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), '/content/drive/MyDrive/converged_scenario_b_transformer.pt')
        print(" -> Saved new optimal model weights checkpoint to Google Drive.")

# 4. Serialize History Logs to JSON
with open('/content/drive/MyDrive/converged_b_history.json', 'w') as f:
    json.dump(history, f)
print("🏁 Core training execution phase complete!")

Executing on device: cuda
Epoch 01/15 | Train Loss: 0.05905 | Val Loss: 0.05144 | LR: 0.000100 | Time: 66.7s
 -> Saved new optimal model weights checkpoint to Google Drive.
Epoch 02/15 | Train Loss: 0.05124 | Val Loss: 0.04980 | LR: 0.000100 | Time: 61.8s
 -> Saved new optimal model weights checkpoint to Google Drive.
Epoch 03/15 | Train Loss: 0.04950 | Val Loss: 0.04904 | LR: 0.000100 | Time: 62.5s
 -> Saved new optimal model weights checkpoint to Google Drive.
Epoch 04/15 | Train Loss: 0.04870 | Val Loss: 0.04857 | LR: 0.000100 | Time: 62.5s
 -> Saved new optimal model weights checkpoint to Google Drive.
Epoch 05/15 | Train Loss: 0.04836 | Val Loss: 0.04835 | LR: 0.000100 | Time: 62.3s
 -> Saved new optimal model weights checkpoint to Google Drive.
Epoch 06/15 | Train Loss: 0.04816 | Val Loss: 0.04808 | LR: 0.000100 | Time: 64.6s
 -> Saved new optimal model weights checkpoint to Google Drive.
Epoch 07/15 | Train Loss: 0.04802 | Val Loss: 0.04804 | LR: 0.000100 | Time: 65.3s
 -> Saved